In [1]:
from lerobot.datasets import LeRobotDataset, LeRobotDatasetMetadata

In [2]:
repo_id = "vtechsavyy7/so101-pick-pen-put-bowl-merged"

metadata = LeRobotDatasetMetadata(repo_id=repo_id)

In [3]:
# Summary of the dataset
print(metadata)

print(metadata.data_path)

LeRobotDatasetMetadata({
    Repository ID: 'vtechsavyy7/so101-pick-pen-put-bowl-merged',
    Total episodes: '68',
    Total frames: '54909',
    Features: '['action', 'observation.state', 'observation.images.camera1', 'observation.images.camera2', 'timestamp', 'frame_index', 'episode_index', 'index', 'task_index']',
})',

data/chunk-{chunk_index:03d}/file-{file_index:03d}.parquet


In [4]:
# Print the episodes info
print(type(metadata.episodes))

# Print the columns of the episodes arrow dataset
print(metadata.episodes.column_names)

# Load the metadata.episodes arrow dataset
episodes_df = metadata.episodes.to_pandas()
# print(episodes_df.head())

<class 'datasets.arrow_dataset.Dataset'>
['episode_index', 'tasks', 'length', 'data/chunk_index', 'data/file_index', 'dataset_from_index', 'dataset_to_index', 'videos/observation.images.camera1/chunk_index', 'videos/observation.images.camera1/file_index', 'videos/observation.images.camera1/from_timestamp', 'videos/observation.images.camera1/to_timestamp', 'videos/observation.images.camera2/chunk_index', 'videos/observation.images.camera2/file_index', 'videos/observation.images.camera2/from_timestamp', 'videos/observation.images.camera2/to_timestamp', 'meta/episodes/chunk_index', 'meta/episodes/file_index']


In [5]:
# Load the dataset itself
dataset = LeRobotDataset(repo_id)
print(f"Number of episodes selected: {dataset.num_episodes}")
print(f"Number of frames selected: {dataset.num_frames}")
print(dataset)

Number of episodes selected: 68
Number of frames selected: 54909
LeRobotDataset({
    Repository ID: 'vtechsavyy7/so101-pick-pen-put-bowl-merged',
    Number of selected episodes: '68',
    Number of selected samples: '54909',
    Features: '['action', 'observation.state', 'observation.images.camera1', 'observation.images.camera2', 'timestamp', 'frame_index', 'episode_index', 'index', 'task_index']',
})


In [16]:
# Load camera images from a sub-section of a given episode

episode_index = 1
from_idx = dataset.meta.episodes["dataset_from_index"][episode_index]
to_idx = dataset.meta.episodes["dataset_to_index"][episode_index]

print(f"Loading camera images from episode {episode_index} (frames {from_idx} to {to_idx})")

camera_key = dataset.meta.camera_keys[0]
frames = [dataset[idx][camera_key] for idx in range(from_idx, to_idx)]

print("Frames number:", len(frames))
print(frames[0].shape)

Loading camera images from episode 1 (frames 899 to 1692)
Frames number: 793
torch.Size([4, 3, 480, 640])


In [17]:
# Using the delta_timestamps feature: 

delta_timestamps = {
        # loads 4 images: 1 second before current frame, 500 ms before, 200 ms before, and current frame
        camera_key: [-1, -0.5, -0.20, 0],
        # loads 6 state vectors: 1.5 seconds before, 1 second before, ... 200 ms, 100 ms, and current frame
        "observation.state": [-1.5, -1, -0.5, -0.20, -0.10, 0],
        # loads 64 action vectors: current frame, 1 frame in the future, 2 frames, ... 63 frames in the future
        "action": [t / dataset.fps for t in range(64)],
    }

dataset = LeRobotDataset(repo_id, delta_timestamps=delta_timestamps)
print(dataset[1670].keys())
print(dataset[1670]["observation.state"])  # (6, state_dim)
print(dataset[1670]["observation.state_is_pad"])
print(dataset[1670]["action"])  # (64, action_dim)
print(dataset[1670]["action_is_pad"])  # (64,)

dict_keys(['action', 'observation.state', 'timestamp', 'frame_index', 'episode_index', 'index', 'task_index', 'observation.images.camera1_is_pad', 'observation.state_is_pad', 'action_is_pad', 'observation.images.camera1', 'observation.images.camera2', 'task'])
tensor([[   6.3736, -101.0110,   96.3077,   55.7802,   -1.3626,   17.1445],
        [   6.3736, -101.0110,   96.3077,   55.7802,   -1.3626,   17.1445],
        [   6.3736, -101.0110,   96.3077,   55.7802,   -1.3626,   17.1445],
        [   6.3736, -101.0110,   96.3077,   55.7802,   -1.3626,   17.1445],
        [   6.3736, -101.0110,   96.3077,   55.7802,   -1.3626,   17.1445],
        [   6.3736, -101.0110,   96.3077,   55.7802,   -1.3626,   17.1445]])
tensor([False, False, False, False, False, False])
tensor([[   6.6374, -104.1319,   96.3516,   55.9121,   -1.7143,   16.7988],
        [   6.6374, -104.1319,   96.3516,   55.9121,   -1.7143,   16.7988],
        [   6.6374, -104.1319,   96.3516,   55.9121,   -1.7143,   16.7988],
   